# 🦀 Day 3: FFI Basics — Calling C from Rust
---

Today we explore **FFI** (Foreign Function Interface) — calling C code from Rust.

- **What FFI is**: Bridging Rust and other languages (especially C)
- **`extern "C"`**: Declaring C-compatible functions
- **C types**: `c_int`, `c_char`, `c_void` from `std::ffi`
- **`CString` and `CStr`**: C-compatible string types
- **Linking**: `#[link(name = "...")]`
- **Safety wrappers**: Hiding unsafe behind safe APIs

In [ ]:
// C types in Rust (std::ffi or std::os::raw)
use std::ffi::{c_int, c_char, c_void};

// Declaring an extern C function (we'll use libc in a real project)
// For EvCxR, we simulate by defining a "mock" C function in Rust
// In a real Cargo project you'd have:
// #[link(name = "m")]  // math library
// extern "C" { fn sqrt(x: f64) -> f64; }

// Simulated C function for demo (in real FFI this comes from a .so/.dll)
// We define it in Rust to mimic what a C library would provide
#[no_mangle]
pub extern "C" fn c_abs(n: c_int) -> c_int {
    if n < 0 { -n } else { n }
}

// To call an EXTERNAL C function, you'd declare:
// extern "C" { fn c_abs(n: c_int) -> c_int; }
// Then: unsafe { c_abs(x) }
let x: c_int = -42;
let result = unsafe { c_abs(x) };
println!("c_abs(-42) = {}", result);

In [ ]:
// CString: Rust String -> C-compatible null-terminated string
// CStr: wrap *const c_char for safe Rust &str conversion
use std::ffi::{CString, CStr};

// Creating a CString from Rust str (adds null terminator)
let rust_str = "hello, FFI!";
let c_string = CString::new(rust_str).expect("CString::new failed");
let ptr = c_string.as_ptr();

// CStr: from raw pointer to &str (unsafe — ptr must be valid)
unsafe {
    let c_str = CStr::from_ptr(ptr);
    let rust_str_back: &str = c_str.to_str().unwrap();
    println!("Round-trip: {}", rust_str_back);
}

In [ ]:
// Safety wrapper: hide unsafe behind a safe API
fn safe_abs(n: i32) -> i32 {
    unsafe { c_abs(n) }
}

println!("safe_abs(-100) = {}", safe_abs(-100));

// Null pointer handling: C often uses null for "error" or "absent"
// Always check ptr.is_null() before dereferencing!

## Cargo project: Calling libc sqrt

In a real project, add to `Cargo.toml`:
```toml
[dependencies]
libc = "0.2"
```

Then in `src/main.rs`:
```rust
use std::ffi::c_double;

#[link(name = "m")]
extern "C" {
    fn sqrt(x: c_double) -> c_double;
}

fn main() {
    let x = 16.0;
    let result = unsafe { sqrt(x) };
    println!("sqrt(16) = {}", result);
}
```

On Windows you may need `#[link(name = "msvcrt")]` or use the `libc` crate which handles platform differences.

## 📝 Exercise: Call a C math function from Rust

Create a Cargo project that calls `pow(base, exp)` from libm. Use `#[link(name = "m")]` on Unix. Write a safe wrapper `fn safe_pow(base: f64, exp: f64) -> f64`.

In [ ]:
// YOUR CODE HERE — create a Cargo project and add the FFI bindings

## 🎯 Key Takeaways

1. **`extern "C"`** declares C-compatible function signatures
2. **C types**: `c_int`, `c_char`, `c_double`, `c_void` from `std::ffi`
3. **`CString`** converts Rust `String`/`&str` to null-terminated C strings
4. **`CStr`** wraps `*const c_char` for safe `&str` conversion
5. **Always check null** before dereferencing C pointers
6. **Safety wrappers** hide `unsafe` from your API users

---
**Tomorrow:** Exposing Rust to C